# 02 - Exploratory Data Analysis and Statistical Analysis
**Syllabus link:** Correlations, data preprocessing

Contents: class distribution, Amount analysis, time pattern, correlations, feature distributions, hypothesis tests.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_style("whitegrid")
FIG_DIR = "../reports/figures"
os.makedirs(FIG_DIR, exist_ok=True)

def savefig(name):
    plt.savefig(f"{FIG_DIR}/{name}.png", dpi=150, bbox_inches="tight")

df = pd.read_csv("../data/processed/creditcard_clean.csv")
df["Class_label"] = df["Class"].map({0: "Legit", 1: "Fraud"})
print(df.shape)

## 1. Class distribution

In [ ]:
vc = df["Class"].value_counts()
pct = df["Class"].value_counts(normalize=True) * 100
print(pd.DataFrame({"count": vc, "percent": pct.round(3)}))

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
sns.countplot(x="Class_label", data=df, ax=ax[0])
ax[0].set_yscale("log")
ax[0].set_title("Class counts (log scale)")
for p in ax[0].patches:
    ax[0].annotate(int(p.get_height()), (p.get_x() + p.get_width() / 2, p.get_height()),
                   ha="center", va="bottom")
ax[1].pie(vc, labels=["Legit", "Fraud"], autopct="%.2f%%", startangle=90, explode=(0, 0.2))
ax[1].set_title("Class share")
plt.tight_layout(); savefig("02_class_distribution"); plt.show()

**Finding:** fraud is a tiny fraction of all transactions, so the data is extremely imbalanced. A model that always predicts "legit" would reach about 99.8% accuracy while catching zero fraud. Hence recall, precision, F1 and PR-AUC are used instead of accuracy.

## 2. Transaction amount by class

In [ ]:
print(df.groupby("Class_label")["Amount"].describe().round(2))

fig, ax = plt.subplots(1, 3, figsize=(17, 4))
for lbl in ["Legit", "Fraud"]:
    sns.kdeplot(np.log1p(df.loc[df["Class_label"] == lbl, "Amount"]), label=lbl, fill=True, ax=ax[0])
ax[0].set_title("log(1+Amount) density"); ax[0].legend()
sns.boxplot(x="Class_label", y=np.log1p(df["Amount"]), data=df, ax=ax[1])
ax[1].set_ylabel("log(1+Amount)"); ax[1].set_title("Boxplot of log(1+Amount)")
sns.violinplot(x="Class_label", y=np.log1p(df["Amount"]), data=df, ax=ax[2])
ax[2].set_ylabel("log(1+Amount)"); ax[2].set_title("Violin plot")
plt.tight_layout(); savefig("02_amount_by_class"); plt.show()

**Finding:** compare the medians and spread in the table and plots above. Write one or two sentences on whether fraud amounts are typically smaller or larger than legitimate ones.

## 3. Fraud by hour of day

In [ ]:
hourly = df.groupby("Hour")["Class"].agg(total="count", fraud="sum")
hourly["fraud_rate_pct"] = hourly["fraud"] / hourly["total"] * 100

fig, ax = plt.subplots(1, 3, figsize=(17, 4))
ax[0].bar(hourly.index, hourly["total"]); ax[0].set_title("All transactions by hour")
ax[1].bar(hourly.index, hourly["fraud"], color="crimson"); ax[1].set_title("Fraud count by hour")
ax[2].plot(hourly.index, hourly["fraud_rate_pct"], marker="o", color="darkred")
ax[2].set_title("Fraud rate (%) by hour")
for a in ax: a.set_xlabel("Hour of day")
plt.tight_layout(); savefig("02_fraud_by_hour"); plt.show()

hourly.sort_values("fraud_rate_pct", ascending=False).head(5)

**Finding:** transaction volume drops during night hours, while the fraud *rate* can rise because legitimate traffic is low. Check the top hours in the table above. Note: the data covers only two days, so treat hourly patterns as indicative.

## 4. Correlation analysis

In [ ]:
num_cols = [col for col in df.columns if col not in ["Class_label", "Hour"]]
corr = df[num_cols].corr()

plt.figure(figsize=(14, 11))
sns.heatmap(corr, cmap="coolwarm", center=0, square=True, cbar_kws={"shrink": 0.7})
plt.title("Correlation heatmap")
savefig("02_corr_heatmap"); plt.show()

In [ ]:
corr_target = corr["Class"].drop("Class").sort_values()

plt.figure(figsize=(8, 8))
corr_target.plot(kind="barh", color=["crimson" if v < 0 else "steelblue" for v in corr_target])
plt.title("Correlation of each feature with Class")
plt.xlabel("Pearson (point-biserial) correlation")
savefig("02_corr_with_class"); plt.show()

top6 = corr_target.abs().sort_values(ascending=False).head(6).index.tolist()
print("Top 6 features by |correlation| with Class:", top6)

**Finding:** `V1-V28` are PCA outputs, so they are almost uncorrelated with each other (the heatmap is mostly blank apart from `Time` and `Amount`). A handful of V-features show a clear correlation with `Class`; these are the strongest candidates for prediction.

## 5. Distribution of top features: fraud vs legit

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for a, f in zip(axes.ravel(), top6):
    sns.kdeplot(df.loc[df["Class"] == 0, f], label="Legit", fill=True, ax=a, common_norm=False)
    sns.kdeplot(df.loc[df["Class"] == 1, f], label="Fraud", fill=True, ax=a, common_norm=False)
    a.set_title(f); a.legend()
plt.tight_layout(); savefig("02_top_feature_kde"); plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for a, f in zip(axes.ravel(), top6):
    sns.boxplot(x="Class_label", y=f, data=df, ax=a)
    a.set_title(f)
plt.tight_layout(); savefig("02_top_feature_box"); plt.show()

**Finding:** for these features the fraud distribution is clearly shifted from the legitimate one, which is why simple classifiers can separate the classes.

## 6. Statistical tests
Significance level: alpha = 0.05.

### 6.1 Normality of `Amount` (Shapiro-Wilk on a random sample of 5000)

In [ ]:
rng = np.random.RandomState(42)
rows = []
for lbl in ["Legit", "Fraud"]:
    x = df.loc[df["Class_label"] == lbl, "Amount"]
    x = x.sample(min(5000, len(x)), random_state=42)
    w, p = stats.shapiro(x)
    rows.append([lbl, round(w, 4), p, "Normal" if p > 0.05 else "Not normal"])
pd.DataFrame(rows, columns=["Group", "W", "p-value", "Conclusion"])

Amount is not normally distributed, so a non-parametric test (Mann-Whitney U) is the primary comparison. A Welch t-test on log(1+Amount) is added as a cross-check.

### 6.2 Is the transaction amount different for fraud?
- H0: Amount distribution is the same for fraud and legit.
- H1: The distributions differ.

In [ ]:
legit_amt = df.loc[df["Class"] == 0, "Amount"]
fraud_amt = df.loc[df["Class"] == 1, "Amount"]

u, p_mw = stats.mannwhitneyu(fraud_amt, legit_amt, alternative="two-sided")
t, p_t = stats.ttest_ind(np.log1p(fraud_amt), np.log1p(legit_amt), equal_var=False)

print(f"Median Amount - fraud: {fraud_amt.median():.2f}, legit: {legit_amt.median():.2f}")
print(f"Mann-Whitney U: U={u:.1f}, p={p_mw:.4g} ->", "reject H0" if p_mw < 0.05 else "fail to reject H0")
print(f"Welch t-test (log Amount): t={t:.3f}, p={p_t:.4g} ->", "reject H0" if p_t < 0.05 else "fail to reject H0")

### 6.3 Does fraud depend on hour of day? (Chi-square test of independence)
- H0: Fraud and hour of day are independent.
- H1: Fraud depends on the hour of day.

In [ ]:
ct = pd.crosstab(df["Hour"], df["Class"])
chi2, p_chi, dof, expected = stats.chi2_contingency(ct)
print(f"Chi-square = {chi2:.2f}, dof = {dof}, p = {p_chi:.4g}")
print("Cells with expected count < 5:", int((expected < 5).sum()), "of", expected.size)
print("->", "reject H0 (fraud depends on hour)" if p_chi < 0.05 else "fail to reject H0")

n = ct.values.sum()
cramers_v = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))
print(f"Cramer's V (effect size) = {cramers_v:.4f}")

A very small Cramer's V means that even if the result is statistically significant, the practical effect of hour on fraud is weak. Mention this in the report.

### 6.4 Point-biserial correlation of each feature with `Class`

In [ ]:
feat_cols = [col for col in df.columns if col not in ["Class", "Class_label", "Hour"]]
rows = []
for f in feat_cols:
    r, p = stats.pointbiserialr(df["Class"], df[f])
    rows.append([f, r, p])
pb = pd.DataFrame(rows, columns=["Feature", "r", "p-value"])
pb["abs_r"] = pb["r"].abs()
pb = pb.sort_values("abs_r", ascending=False).reset_index(drop=True)
pb["Significant (p<0.05)"] = pb["p-value"] < 0.05
pb.head(10)

### 6.5 Two-sample Kolmogorov-Smirnov test (which features separate the classes best?)

In [ ]:
rows = []
for f in feat_cols:
    ks, p = stats.ks_2samp(df.loc[df["Class"] == 1, f], df.loc[df["Class"] == 0, f])
    rows.append([f, ks, p])
ks_df = pd.DataFrame(rows, columns=["Feature", "KS statistic", "p-value"]).sort_values("KS statistic", ascending=False)
ks_df.head(10)

## Summary of EDA findings (copy into the report)
1. Extreme class imbalance, so accuracy is misleading.
2. `Amount` is skewed and non-normal; fraud and legit amounts differ (see Mann-Whitney result).
3. Fraud rate varies by hour; chi-square tests dependence, Cramer's V gives its size.
4. A few V-features carry most of the class information (correlation and KS ranking agree).
5. Outliers kept, robust scaling chosen.

Next: `03_regression_analysis.ipynb`.